# Synthetic valley advanced model

This notebook builds the coastal synthetic-valley model with the MODFLOW 6 advanced packages, adds two multi-aquifer wells, runs it, and compares the simulated heads to the true heads at the observation wells. It then asks how pumping from the well near the stream changes the water the stream carries out of the model. The companion notebook [`synthetic-valley-a-base`](synthetic-valley-a-base.ipynb) runs the same valley with simple boundary packages.

> **Model reference.** The synthetic valley model is described in Hughes, J.D., Langevin, C.D., Paulinski, S.R., Larsen, J.D., and Brakenhoff, D., 2024, FloPy workflows for creating structured and unstructured MODFLOW models: Groundwater, v. 62, no. 1, p. 124–139, https://doi.org/10.1111/gwat.13327.

## What this notebook covers

By the end of this notebook you will be able to:

- say what the Unsaturated Zone Flow, Streamflow Routing, Lake, Water Mover, and Multi-Aquifer Well packages represent,
- build the Multi-Aquifer Well package with `flopy.mf6.ModflowGwfmaw()`,
- plot the water table in plan view and in cross section and compare simulated heads to observed heads, and
- measure how much a pumping well reduces the discharge of a nearby stream.

## The model

The synthetic valley drains south to the sea. The model uses a structured grid of 40 rows and 25 columns of 152.4 meter (m) cells, and five layers: layers 1 and 2 are the upper aquifer, layer 3 is a clay confining unit present only in part of the valley, and layers 4 and 5 are the lower aquifer. Where there is no clay, the layer 3 cells are **vertical pass-through cells** (`idomain` of -1), which have no volume and connect layer 2 directly to layer 4. The sea is a General-Head Boundary (**GHB**) at a head of 0 m along the southern edge.

The **advanced packages** represent the surface-water features and the land surface as more than a fixed boundary:

- **Unsaturated Zone Flow (UZF)** moves rainfall down through the unsaturated zone to the water table and takes evapotranspiration from it. Where the water table reaches the land surface, the infiltration it cannot accept runs off.
- **Streamflow Routing (SFR)** routes water down the stream, reach by reach, and exchanges water with the aquifer beneath each reach. A **reach** is a length of stream in one cell.
- **Lake (LAK)** keeps a water balance for the lake, so its level rises and falls with rainfall, evaporation, runoff, and leakage to and from the aquifer. The lake occupies inactive cells in layer 1.
- **Water Mover (MVR)** sends the runoff from UZF to the lake or the stream.
- **Multi-Aquifer Well (MAW)** represents a well screened across several layers. It solves for one water level in the wellbore and works out how much water each layer gives up.

The first stress period is a 16,071-day **steady-state** period, which gives the flow system before pumping. Twenty one-year transient periods follow. MODFLOW has no built-in units, so every input shares one length and one time unit: here meters and days.

## Imports

Import FloPy, Matplotlib, NumPy, and pandas. `sv_notebook_helpers` holds the functions that build the model, read the true and simulated heads at the observation wells, and read and save the stream discharge.

In [ ]:
%matplotlib inline
import pathlib as pl

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sv_notebook_helpers import (
    build_advanced_without_maw,
    record_sfr_discharge,
    sfr_discharge,
    simulated_heads,
    true_heads,
)

## Build the model with the advanced packages

`build_advanced_without_maw()` writes the base model to a new workspace with the Recharge, Evapotranspiration, River, and Well packages replaced by UZF, SFR, LAK, and MVR, and loads it with `flopy.mf6.MFSimulation.load()`. It leaves out MAW, which you build next. `pathlib.Path` builds the path to the workspace.

In [ ]:
name = "sv"
new_ws = pl.Path("models/synthetic-valley-advanced")

sim = build_advanced_without_maw(new_ws)
gwf = sim.get_model(name)
gwf.get_package_list()

Read the arrays the wells and figures need: `idomain` and the cell bottoms `botm` from the discretization (**DIS**) package. Each is a NumPy array of shape `(nlay, nrow, ncol)`, and FloPy counts layers, rows, and columns from 0, so `botm[3]` is the bottom of layer 4.

The lake cells are the inactive cells (`idomain` of 0) in layer 1. The cross sections below run through the row and column at the center of the lake.

In [ ]:
idomain = gwf.dis.idomain.array
botm = gwf.dis.botm.array
extent = gwf.modelgrid.extent
nper = sim.tdis.nper.data

lake = idomain[0] == 0
lake_rows, lake_cols = np.nonzero(lake)
row_xs = int(round(lake_rows.mean()))
col_xs = int(round(lake_cols.mean()))
print(
    f"{lake.sum()} lake cells; sections along row {row_xs + 1} and column {col_xs + 1}"
)

## Build the Multi-Aquifer Well package

Two wells pump from the lower aquifer, each screened across layers 4 and 5. The `reilly` well is in the north of the valley, and the `vc` well is three columns (457 m) west of the stream. Set the pumping rate of each well in cubic meters per day (m³/d). A negative rate removes water from the aquifer.

In [ ]:
reilly_rate = -1897.2  # m3/d
vc_rate = -7588.9  # m3/d

# well name: (first layer, last layer, row, column), counted from 0
well_data = {
    "reilly": (3, 4, 5, 14),
    "vc": (3, 4, 32, 5),
}
rates = {"reilly": reilly_rate, "vc": vc_rate}

The package data gives one row per well: its number (counted from 0), the wellbore radius, the bottom of the well, the starting water level, the equation used for the conductance between the well and the aquifer, the number of layers it is connected to, and its name. `"thiem"` computes that conductance from the cell's hydraulic conductivity and the Thiem equation for steady radial flow to a well.

In [ ]:
# (well, radius, bottom, starting head, conductance equation, connections, name)
# (0, 0.1524, -67.2, 0.0, "thiem", 2, "reilly")
package_data = []
for iw, (well, (k0, k1, i, j)) in enumerate(well_data.items()):
    package_data.append((iw, 0.1524, float(botm[k1, i, j]), 0.0, "thiem", 2, well))
package_data

The connection data gives one row per well per layer: the well and connection numbers, the cell `(layer, row, column)`, and the top and bottom of the screen in that layer. Each screen runs the full thickness of its cell, from the bottom of the layer above to the bottom of the layer. The skin values are not used by the Thiem equation, so they are set to -999.

In [ ]:
# (well, connection, (layer, row, column), screen top, screen bottom, skin K, skin radius)
connection_data = []
for iw, (well, (k0, k1, i, j)) in enumerate(well_data.items()):
    for ic, k in enumerate(range(k0, k1 + 1)):
        screen_top = float(botm[k - 1, i, j])
        screen_bottom = float(botm[k, i, j])
        connection_data.append(
            (iw, ic, (k, i, j), screen_top, screen_bottom, -999.0, -999.0)
        )
connection_data

The period data switches the wells off for the steady-state period and on, at their pumping rates, from the second stress period. It is a dictionary keyed by the stress-period number, counted from 0. A setting carries forward to later stress periods until it is changed, so the rates only need to be given once.

In [ ]:
maw_spd = {
    0: [(iw, "status", "inactive") for iw in range(len(well_data))],
    1: [(iw, "status", "active") for iw in range(len(well_data))]
    + [(iw, "rate", rates[well]) for iw, well in enumerate(well_data)],
}

maw = flopy.mf6.ModflowGwfmaw(
    gwf,
    pname="pwell",
    boundnames=True,
    nmawwells=len(well_data),
    packagedata=package_data,
    connectiondata=connection_data,
    perioddata=maw_spd,
)
gwf.get_package_list()

## Run the model

Write the input files with `write_simulation()` and run MODFLOW 6 with `run_simulation()`, which returns `True` when the run ends normally.

In [ ]:
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"

## Water table at the end of the steady-state period

Read the heads at the end of the steady-state period with `gwf.output.head().get_data()`. `kstpkper=(0, 0)` selects the first time step of the first stress period, counting from 0. Inactive cells hold a head of 1.0E+30, which `masked_values` leaves out of the plots.

The lake cells are inactive, so the aquifer has no head there. LAK computes the lake level, its **stage**, instead: read it with `gwf.lak.output.stage().get_data()` and put it in the layer 1 lake cells, so the maps and sections show the lake surface.

Map the head in layer 1, which is the water table, and draw it along a west-east and a north-south cross section through the center of the lake. In the cross sections, `plot_array(head, head=head)` colors each cell only up to the simulated head or lake stage, so the colored area ends at the water table or the lake surface. `plot_ibound()` shades the pass-through cells in layer 3 gray. The sections show the top 25 m of the model, because the upper aquifer is only a few meters thick.

In [ ]:
width = 17.15 / 2.54  # figure width, inches
head = gwf.output.head().get_data(kstpkper=(0, 0))
stage = gwf.lak.output.stage().get_data(kstpkper=(0, 0)).ravel()[0]
head[0][lake] = stage
water_table = np.ma.masked_greater(head[0], 1.0e29)
vmin, vmax = np.floor(water_table.min()), np.ceil(water_table.max())
levels = np.arange(vmin, vmax + 0.5, 0.5)
top = gwf.dis.top.array
x_ns = gwf.modelgrid.xcellcenters[0, col_xs]
y_ew = gwf.modelgrid.ycellcenters[row_xs, 0]
# elevation range of the sections: land surface to 25 m below its highest point
ylim = (np.ceil(top.max()) - 25.0, np.ceil(top.max()))
sections = {
    "B": ({"row": row_xs}, "West-east section", "Distance from the west edge (m)"),
    "C": (
        {"column": col_xs},
        "North-south section",
        "Distance from the north edge (m)",
    ),
}

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"], ["A", "C"]],
        figsize=(width, 0.85 * width),
        layout="constrained",
        width_ratios=[1.0, 1.5],
    )
    ax = axd["A"]
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    pa = mm.plot_array(head, masked_values=[1.0e30], vmin=vmin, vmax=vmax)
    cs = mm.contour_array(
        head, masked_values=[1.0e30], levels=levels, colors="black", linewidths=0.5
    )
    ax.clabel(cs, fmt="%g", fontsize=6)
    mm.contour_array(lake.astype(float), levels=[0.5], colors="blue", linewidths=1.0)
    mm.plot_bc(package=gwf.sfr, color="cyan")
    mm.plot_bc(package=gwf.ghb, color="purple", kper=0)
    mm.plot_bc(package=maw, color="red", kper=1)
    ax.plot(extent[:2], [y_ew, y_ew], color="red", lw=1.0, ls="--")
    ax.plot([x_ns, x_ns], extent[2:], color="red", lw=1.0, ls="--")
    ax.text(extent[0], y_ew, "B ", color="red", ha="right", va="center")
    ax.text(x_ns, extent[3], "C", color="red", ha="center", va="bottom")
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    flopy.plot.styles.heading(ax=ax, letter="A", heading="Water table")
    fig.colorbar(pa, ax=ax, orientation="horizontal", shrink=0.9, label="Head (m)")

    for letter, (line, heading, xlabel) in sections.items():
        ax = axd[letter]
        xs = flopy.plot.PlotCrossSection(model=gwf, ax=ax, line=line)
        # draw the inactive cells first, so the lake cells take the lake stage
        xs.plot_ibound(color_noflow="white", color_vpt="0.85")
        xs.plot_array(head, head=head, masked_values=[1.0e30], vmin=vmin, vmax=vmax)
        xs.plot_grid(lw=0.25, color="0.5")
        xs.plot_surface(top, color="saddlebrown", lw=1.0)
        xs.plot_surface(water_table, color="blue", lw=1.0)
        ax.set_ylim(ylim)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Elevation (m)")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=heading)
    handles = [
        mpl.lines.Line2D([], [], color="saddlebrown", lw=1.0, label="Land surface"),
        mpl.lines.Line2D(
            [], [], color="blue", lw=1.0, label="Water table and lake surface"
        ),
    ]
    flopy.plot.styles.graph_legend(
        ax=axd["C"],
        handles=handles,
        labels=[h.get_label() for h in handles],
        loc="upper center",
        bbox_to_anchor=(0.5, -0.3),
        ncol=2,
    )

In [ ]:
aquifer = np.ma.masked_where(lake, water_table)
print(f"water table from {aquifer.min():.2f} to {aquifer.max():.2f} m")
print(f"lake stage {stage:.2f} m")

**What to look for.** The water table (A) falls from 4.11 m in the northeast to -0.15 m in the stream cells nearest the coast, where the streambed is below sea level. The lake stage is 4.30 m, higher than the water table around it, so the lake leaks to the aquifer. The contours bend upstream along the stream (cyan), which gains water from the upper aquifer along its length. In the sections (B and C) the lake fills its cells up to its stage, the water table (blue) lies within a few meters of the land surface (brown), and the gray cells in layer 3 are where the clay is absent. The wells (red) are switched off in this period, so the heads are those of the valley before pumping.

## Compare the simulated heads to the true heads

The observation wells measure the water table (layer 1) and the lower aquifer (layers 4 and 5) at 17 locations. The true heads come from the Voronoi-grid advanced model in `examples/data/synthetic-valley/data`, which defines the truth for the class. `true_heads()` returns them and `simulated_heads()` reads the simulated heads the model saved at the same wells, each as a table with one column per well and one row per output time. Select the end of the steady-state period with `.loc[]`.

A **residual** is the simulated head minus the true head. A positive residual means the model is too high.

In [ ]:
t_steady = gwf.output.head().get_times()[0]
kinds = {"A": ("wt", "Water table"), "B": ("aq", "Lower aquifer")}
residuals = {}
for letter, (kind, label) in kinds.items():
    true = true_heads(kind).loc[t_steady]
    sim_head = simulated_heads(new_ws, kind).loc[t_steady, true.index]
    r = sim_head - true
    residuals[letter] = (true, sim_head, r)
    rmse = np.sqrt((r**2).mean())
    print(
        f"{label}: true heads {true.min():.3f} to {true.max():.3f} m; "
        f"mean residual {r.mean():.3f} m, root-mean-square {rmse:.3f} m, "
        f"largest {r.abs().max():.3f} m"
    )

Map the residuals at each well, colored from blue (simulated head too low) to red (too high), and plot the simulated heads with the true heads on a one-to-one plot, where a well whose simulated head equals its true head falls on the diagonal line. The observation names hold the row and column of each well counted from 0, so `aq05-i14-j11` is in row 15 and column 12; `obs_cell()` reads them back.

In [ ]:
def obs_cell(obsname):
    # "AQ05-I14-J11" -> (13, 10)
    _, i, j = obsname.split("-")
    return int(i[1:]), int(j[1:])


rmax = max(r.abs().max() for _, _, r in residuals.values())
rnorm = mpl.colors.Normalize(vmin=-rmax, vmax=rmax)
values = np.concatenate([np.r_[t, s] for t, s, _ in residuals.values()])
lim = (np.floor(values.min()), np.ceil(values.max()))
markers = {"A": "o", "B": "s"}

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [["A", "B", "C"]],
        figsize=(width, 0.55 * width),
        layout="constrained",
        width_ratios=[1.0, 1.0, 1.4],
    )
    for letter, (kind, label) in kinds.items():
        true, sim_head, r = residuals[letter]
        ax = axd[letter]
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
        mm.plot_array(head, masked_values=[1.0e30], vmin=vmin, vmax=vmax, alpha=0.4)
        mm.contour_array(
            lake.astype(float), levels=[0.5], colors="blue", linewidths=1.0
        )
        mm.plot_bc(package=gwf.sfr, color="cyan")
        cells = [obs_cell(n) for n in r.index]
        x = [gwf.modelgrid.xcellcenters[i, j] for i, j in cells]
        y = [gwf.modelgrid.ycellcenters[i, j] for i, j in cells]
        sc = ax.scatter(
            x,
            y,
            c=r,
            cmap="RdBu_r",
            norm=rnorm,
            marker=markers[letter],
            s=30,
            edgecolors="black",
            linewidths=0.5,
            zorder=10,
        )
        ax.set_aspect("equal")
        ax.set_xlabel("x (m)")
        if letter == "A":
            ax.set_ylabel("y (m)")
        flopy.plot.styles.heading(ax=ax, letter=letter, heading=f"{label} residual")
    fig.colorbar(
        sc,
        ax=[axd["A"], axd["B"]],
        orientation="horizontal",
        shrink=0.6,
        label="Simulated minus true head (m)",
    )

    ax = axd["C"]
    ax.plot(lim, lim, color="0.5", lw=0.75, label="One-to-one line")
    for letter, (kind, label) in kinds.items():
        true, sim_head, _ = residuals[letter]
        ax.plot(
            true,
            sim_head,
            markers[letter],
            ms=4,
            mfc="none",
            color="black",
            label=label,
        )
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_aspect("equal")
    flopy.plot.styles.heading(ax=ax, letter="C", heading="Simulated and true heads")
    flopy.plot.styles.xlabel(ax=ax, label="True head (m)")
    flopy.plot.styles.ylabel(ax=ax, label="Simulated head (m)")
    flopy.plot.styles.graph_legend(ax=ax, loc="lower right", fontsize=7)

**What to look for.** The simulated heads are 0.05 m low on average at the water table and 0.06 m low in the lower aquifer, the root-mean-square residual is 0.08 m for both, and no well is off by more than 0.21 m. In the maps (A and B) the wells are pale, except for the dark blue well east of the stream in the southern half of the valley, which carries the largest residual in both aquifers. On the one-to-one plot (C) every well lies within 0.21 m of the line.

## Question: how does the well near the stream change the stream discharge?

The `vc` well pumps from the lower aquifer 457 m west of the stream. Some of the water it pumps would otherwise have discharged to the stream, so pumping lowers the flow the stream carries out of the model to the sea. That reduction is called **streamflow depletion**.

**What effect does pumping from the `vc` well have on the surface-water discharge from the model?** To answer it:

1. Run the notebook as it is, which records the discharge for the current `vc_rate`.
2. In the cell that sets the pumping rates, change `vc_rate`, for example to `0.0` (no pumping) or to twice the current rate.
3. Run that cell and every cell below it again. `build_advanced_without_maw()` does not need to run again, but the cells from **Build the Multi-Aquifer Well package** down do.
4. Repeat for as many rates as you like, then compare the runs in the figure at the end of this section.

`sfr_discharge()` reads the flow out of the last reach of the stream at the end of each stress period, in m³/d. `record_sfr_discharge()` keeps the runs in a dictionary whose key is the `vc` pumping rate and whose value is the discharge, replacing any earlier run at the same rate. It saves the dictionary to `models/sfr-discharge.pkl` with Python's `pickle` module, which writes a Python object to a binary file and reads it back unchanged, so the runs survive restarting the notebook. `pd.DataFrame(runs)` turns the dictionary into a table with one column per pumping rate.

In [ ]:
discharge = sfr_discharge(new_ws)
runs = record_sfr_discharge("models/sfr-discharge.pkl", vc_rate, discharge)
table = pd.DataFrame(runs)
table

Plot the stream discharge through time for each saved run (A), and the average discharge over the pumping period for each `vc` pumping rate (B). The time axis is the end of each stress period in years since the start of pumping.

In [ ]:
years = (table.index - table.index[0]) / 365.25
pumping = table.index > table.index[0]
mean_discharge = table[pumping].mean()

with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B"]],
        figsize=(width, 0.4 * width),
        layout="constrained",
        width_ratios=[2.0, 1.0],
    )
    ax = axd["A"]
    for rate in table.columns:
        ax.plot(years, table[rate], marker="o", ms=3, label=f"{rate:g} m³/d")
    ax.set_xlim(years.min(), years.max())
    # room above the lines for the explanation
    ax.set_ylim(0.0, 1.4 * table.to_numpy().max())
    flopy.plot.styles.heading(ax=ax, letter="A", heading="Stream discharge to the sea")
    flopy.plot.styles.xlabel(ax=ax, label="Years since pumping started")
    flopy.plot.styles.ylabel(ax=ax, label="Discharge (m³/d)")
    flopy.plot.styles.graph_legend(
        ax=ax,
        loc="upper center",
        ncol=3,
        title="vc pumping rate",
        columnspacing=1.0,
        handlelength=1.5,
    )

    ax = axd["B"]
    ax.plot(-mean_discharge.index, mean_discharge, marker="o", color="black")
    ax.set_ylim(0.0, 1.1 * mean_discharge.max())
    flopy.plot.styles.heading(ax=ax, letter="B", heading="Average while pumping")
    flopy.plot.styles.xlabel(ax=ax, label="vc pumping (m³/d)")
    flopy.plot.styles.ylabel(ax=ax, label="Discharge (m³/d)")

**Hints for your answer.** Compare the average discharge in panel B for two pumping rates. The difference divided by the difference in pumping is the fraction of the pumped water that comes from the stream. Does that fraction change with the pumping rate? Does the stream take a year, or several, to feel a change in pumping (panel A)? Where else could the rest of the pumped water come from?

## Recap

You built the coastal synthetic-valley model with the UZF, SFR, LAK, and MVR packages, added the two pumping wells with the Multi-Aquifer Well package, ran the model, drew the steady-state water table in plan view and along two cross sections through the lake, and compared the simulated heads at 17 observation wells to the true heads on a map and a one-to-one plot. You then measured how the stream discharge to the sea changes with the pumping rate of the well near the stream.